# 02 - Data Validation**Objective:** Validate income records using Pydantic models and Pandera DataFrame schemas.**Tools:** Pydantic (row-level), Pandera (DataFrame-level)**Steps:**1. Define a Pydantic model for income records2. Implement row-level validation3. Define a Pandera DataFrame schema4. Implement DataFrame-level validation5. Test on clean and corrupted data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathfrom pydantic import BaseModel, field_validatorimport pandera as pafrom pandera.typing import DataFrame, Seriesprint("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data from data/processed/clean_train.csv# We'll validate this data using both Pydantic and Pandera.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_train.csv")# print(f"Data shape: {df.shape}")# print(f"Columns: {list(df.columns)}")

### Pydantic Model DefinitionPydantic validates data at the **record level** — each row is checked individually.Define a model with typed fields and add `@field_validator` decorators to enforcebusiness rules like valid age ranges, realistic work hours, etc.This catches row-level issues like a negative age, an income value of 2,or someone working 200 hours per week that might slip through type-only checks.

In [ ]:
# === Executed Example: Pydantic Row-Level Validation ===# This cell uses a small inline dataset (no CSV needed) to demonstrate# how Pydantic catches invalid rows at the record level.import pandas as pdfrom pydantic import BaseModel, field_validatordata = pd.DataFrame({    "id": [1, 2, 3, 4, 5],    "age": [39, -5, 25, 999, 42],    "hours_per_week": [40, 30, 200, 50, -10],})class Record(BaseModel):    id: int    age: int    hours_per_week: float    @field_validator("age")    @classmethod    def age_in_range(cls, v):        if not (0 <= v <= 120):            raise ValueError(f"Age out of range: {v}")        return v    @field_validator("hours_per_week")    @classmethod    def hours_in_range(cls, v):        if not (1 <= v <= 99):            raise ValueError(f"Hours per week out of range: {v}")        return vvalid = []for _, row in data.iterrows():    try:        Record(**row.to_dict())        valid.append(True)    except ValueError:        valid.append(False)print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [ ]:
# === Commented Template: Pydantic Row-Level Validation ===# Copy, paste, and adapt to your own dataset. Uncomment to use.# import pandas as pd# from pydantic import BaseModel, field_validator# data = pd.DataFrame({#     "field_1": [val1, val2, val3],#     "field_2": [val1, val2, val3],# })# class MyRecord(BaseModel):#     field_1: int#     field_2: float#     @field_validator("field_1")#     @classmethod#     def rule_name(cls, v):#         if not (lower <= v <= upper):#             raise ValueError(f"Constraint description: {v}")#         return v# valid = []# for _, row in data.iterrows():#     try:#         MyRecord(**row.to_dict())#         valid.append(True)#     except ValueError:#         valid.append(False)# print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [ ]:
# TODO: Define the Pydantic model for income records# Each field needs a type hint. Add field_validators to enforce constraints.## Validation rules you might want:#   - income must be 0 or 1#   - age should be between 0 and 120#   - hours-per-week should be between 1 and 99#   - capital-gain and capital-loss should be non-negative#   - education-num should be between 1 and 16class IncomeRecord(BaseModel):    """Pydantic model for a single income record."""    # TODO: Add all fields with proper type hints    # age: float    # fnlwgt: float    # education-num: float    # capital-gain: float    # capital-loss: float    # hours-per-week: float    # income: int    # (plus one-hot encoded columns for workclass, education,    #  marital-status, occupation, relationship, race, sex, native-country)    # TODO: Add field validators for business rules    #    # @field_validator("income")    # def check_income(cls, v):    #     """Income must be 0 (<=50K) or 1 (>50K)."""    #     if v not in [0, 1]:    #         raise ValueError(f"Income must be 0 or 1, got {v}")    #     return v    pass

In [ ]:
# TODO: Implement a function that validates a DataFrame row-by-row using Pydantic# This function iterates over each row, tries to construct an IncomeRecord,# and collects whether the row passed or failed validation.def validate_with_pydantic(df):    """Validate each row of the DataFrame using the Pydantic model.    Returns a list of booleans where True means the row is valid.    Also prints a summary of how many rows passed vs failed.    """    # TODO: Loop through each row with df.iterrows()    # For each row, try:    #     IncomeRecord(**row.to_dict())    # If it succeeds, mark the row as valid (True), otherwise invalid (False)    #    # valid = []    # for _, row in df.iterrows():    #     try:    #         IncomeRecord(**row.to_dict())    #         valid.append(True)    #     except Exception:    #         valid.append(False)    #    # valid_count = sum(valid)    # invalid_count = len(valid) - valid_count    # print(f"Valid: {valid_count}, Invalid: {invalid_count}")    # return valid    pass# TODO: Test the Pydantic validation on the clean data# valid_flags = validate_with_pydantic(df)# print(f"Pass rate: {sum(valid_flags) / len(valid_flags):.1%}")

### Pandera Schema DefinitionPandera validates data at the **DataFrame level** — instead of checking one row at a time,it defines column-wide constraints (data types, value ranges, nullable rules, etc.).This is more efficient for bulk validation and catches issues like a columnthat unexpectedly contains strings instead of numbers.

In [ ]:
# === Executed Example: Pandera DataFrame-Level Validation ===# This cell uses the same inline dataset to demonstrate how Pandera# validates all rows at once, collecting all failures.import pandas as pdimport pandera as padata = pd.DataFrame({    "id": [1, 2, 3, 4, 5],    "age": [39, -5, 25, 999, 42],    "hours_per_week": [40, 30, 200, 50, -10],})Schema = pa.DataFrameSchema({    "id": pa.Column(pa.Int, nullable=False),    "age": pa.Column(pa.Int, checks=[pa.Check.ge(0), pa.Check.le(120)]),    "hours_per_week": pa.Column(pa.Float, checks=[pa.Check.ge(1), pa.Check.le(99)]),})try:    Schema.validate(data, lazy=True)    print("Pandera: all rows passed")except pa.errors.SchemaErrors as e:    print(f"Pandera: {len(e.failure_cases)} failures found")    print(e.failure_cases[["index", "column", "check", "failure_case"]].head())

In [ ]:
# === Commented Template: Pandera DataFrame-Level Validation ===# Copy, paste, and adapt to your own dataset. Uncomment to use.# import pandas as pd# import pandera as pa# data = pd.DataFrame({#     "field_1": [val1, val2, val3],#     "field_2": [val1, val2, val3],# })# MySchema = pa.DataFrameSchema({#     "field_1": pa.Column(pa.Int, nullable=False),#     "field_2": pa.Column(pa.Float, checks=[pa.Check.ge(min_val), pa.Check.le(max_val)]),# })# try:#     MySchema.validate(data, lazy=True)#     print("Pandera: all rows valid")# except pa.errors.SchemaErrors as e:#     print(f"Pandera: {len(e.failure_cases)} failure(s)")

In [ ]:
# TODO: Define a Pandera DataFrameSchema for income data# Each column gets a pa.Column with a dtype and optional Check constraints.## For example:#   "income": pa.Column(pa.Int, checks=pa.Check.isin([0, 1]))# ensures the income column only contains 0 or 1.## Numeric columns like "age" and "hours-per-week" need range checks:#   "age": pa.Column(pa.Float, checks=[pa.Check.ge(0), pa.Check.le(120)])# IncomeSchema = pa.DataFrameSchema({#     "income": pa.Column(pa.Int, checks=pa.Check.isin([0, 1])),#     "age": pa.Column(pa.Float, checks=[pa.Check.ge(0), pa.Check.le(120)]),#     "hours-per-week": pa.Column(pa.Float, checks=[pa.Check.ge(1), pa.Check.le(99)]),#     "capital-gain": pa.Column(pa.Float, checks=pa.Check.ge(0)),#     "capital-loss": pa.Column(pa.Float, checks=pa.Check.ge(0)),#     # TODO: Add the rest of the columns with appropriate checks#     # ...# })

In [ ]:
# TODO: Implement a function that validates a DataFrame using the Pandera schema# Call IncomeSchema.validate(df) which raises an exception if validation fails.# Wrap it in a try/except to handle the error gracefully.def validate_with_pandera(df):    """Validate the entire DataFrame using the Pandera schema."""    # TODO: Call IncomeSchema.validate(df) inside a try block    # If it succeeds, print "Pandera validation passed"    # If it raises an exception, print the error message    pass# TODO: Test the Pandera validation on the clean data# validate_with_pandera(df)

In [ ]:
# Exercise: Test validation on corrupted data# The data_injection module can create corrupted variants of the clean data.# Run both Pydantic and Pandera validation on corrupted data and observe# what kinds of issues each validation method catches.## TODO: Load corrupted data and run both validators# from data_injection.injector import load_and_inject# df_corrupted = load_and_inject("../data/processed/clean_train.csv", preset="missing_heavy")## valid_flags = validate_with_pydantic(df_corrupted)# print(f"Corrupted data pass rate: {sum(valid_flags) / len(valid_flags):.1%}")## validate_with_pandera(df_corrupted)

### Exercises1. **Add more validators**: Add a `@field_validator` for `hours-per-week` that rejects values below 1 or above 99 (standard full-time range).2. **Compare error messages**: Run the same corrupted data through both Pydantic and Pandera — which one gives more informative error messages?3. **Performance test**: Time both validators on the full dataset (use `%%timeit` in a cell) — is Pandera noticeably faster for bulk validation?4. **Custom error handler**: Modify `validate_with_pydantic()` to collect and return the actual error messages (not just True/False) so students can see why a row failed.5. **Schema evolution**: What happens if a new column is added to the CSV? Will Pandera reject it or pass it through? Check the `strict` parameter in `pa.DataFrameSchema`.